# §1.2.2 — 손실을 바꾸면 최적해가 바뀐다

> 딥러닝 교재 · 1부 1장 2절 4항 (🐍)
> 선행: §1.2.1(점별 최소화) · §1.2.2(세 폐형식) · §1.2.3(두 가우시안 손계산)
> 부록 J(표기) · 부록 K(노트북 사용 안내)

## 이 노트북이 답하는 질문

1. **§1.2.2의 세 폐형식이 실제로 다른 값을 주는가?** 참값을 폐형식으로 아는 비대칭 분포에서 확인한다.
2. **각 추정량은 자기 목표에만 수렴하는가?** MSE로 적합한 것과 중앙값의 거리는 $n$을 늘리면 줄어드는가, 줄지 않는가.
3. **비대칭이 심할수록 격차가 커지는가?** 한 개의 손잡이로 확인한다.
4. **조건부 분포가 다봉이면 무슨 일이 일어나는가?** 제곱오차의 최적해가 실제로 밀도가 거의 없는 자리에 놓이는 것을 본다.

**예상 실행 시간** CPU 단일 코어 약 45초 (`FAST = True`이면 약 15초).

이 노트북은 모형이나 최적화를 다루지 않습니다. **손실만 바꾸고 나머지를 전부 고정**했을 때 답이 달라지는 것만 봅니다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

_t_start = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────────
FAST     = False
SEED     = 20260731
S        = 0.8      # 비대칭 강도. 로그정규의 sigma. 0에 가까우면 세 값이 겹친다
SAVE_PDF = False    # True면 figs/ 에 벡터 PDF로 저장
FIG_DIR  = 'figs'
# ────────────────────────────────────────────────────────────

CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
    'figure.autolayout': True,
})

import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_avail = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _avail), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

def lab(ko, en):
    return ko if KO_FONT else en

_fig_i = [0]
def show(name):
    _fig_i[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_1_2_4_{_fig_i[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | S={S} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 참값을 폐형식으로 아는 비대칭 분포

**자료 생성 분포.** $x$는 균등, $y$는 조건부로 로그정규를 따른다.

$$x \sim \mathrm{Uniform}(0,1), \qquad y \mid x \sim \mathrm{LogNormal}\big(\mu(x), s\big), \qquad \mu(x) = 1 + 0.8\sin(2\pi x)$$

로그정규를 고른 이유는 하나다 — **세 요약통계가 모두 닫힌 형태로 나오고, 셋 다 다르다.**

$$\mathbb{E}[y \mid x] = e^{\mu(x) + s^2/2}, \qquad \mathrm{median}(y \mid x) = e^{\mu(x)}, \qquad \mathrm{mode}(y \mid x) = e^{\mu(x) - s^2}$$

세 곡선이 서로 상수배라는 점이 편리하다. 격차가 $s$ 하나로 조절된다.

$$\frac{\text{평균}}{\text{중앙값}} = e^{s^2/2}, \qquad \frac{\text{최빈값}}{\text{중앙값}} = e^{-s^2}$$

§1.2.2에 의해 이 셋이 각각 **제곱오차 · 절대오차 · 0-1 손실**의 최적해다.

In [ ]:
def mu(x):
    return 1.0 + 0.8*np.sin(2*np.pi*np.asarray(x))

def true_mean(x):   return np.exp(mu(x) + S**2/2)
def true_median(x): return np.exp(mu(x))
def true_mode(x):   return np.exp(mu(x) - S**2)

def sample(n, rng):
    x = rng.uniform(0, 1, size=n)
    y = np.exp(mu(x) + S*rng.normal(size=n))
    return x, y

print(f"평균/중앙값 = {np.exp(S**2/2):.4f}배,   최빈값/중앙값 = {np.exp(-S**2):.4f}배\n")
print("  x       평균     중앙값    최빈값")
for xv in [0.0, 0.25, 0.5, 0.75]:
    print(f"{xv:5.2f}  {true_mean(xv):7.4f}  {true_median(xv):7.4f}  {true_mode(xv):7.4f}")

In [ ]:
X0 = 0.25                                   # 이하 '한 입력'으로 고정해 쓸 지점
yy = np.linspace(0.01, 30, 800)
dens = (1/(yy*S*np.sqrt(2*np.pi)))*np.exp(-(np.log(yy)-mu(X0))**2/(2*S**2))

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(yy, dens, color=CB[0], lw=1.6)
ax.fill_between(yy, dens, color=CB[2], alpha=0.18)
for v, c, nm_ko, nm_en in [(true_mode(X0),   CB[1], '최빈값 (0-1 손실)',   'mode (0-1 loss)'),
                           (true_median(X0), CB[3], '중앙값 (절대오차)',   'median (abs. loss)'),
                           (true_mean(X0),   CB[4], '평균 (제곱오차)',     'mean (squared loss)')]:
    ax.axvline(v, color=c, lw=2, label=lab(f'{nm_ko}  {v:.2f}', f'{nm_en}  {v:.2f}'))
ax.set_xlim(0, 25)
ax.set_xlabel(lab('$y$ (출력값)', '$y$'))
ax.set_ylabel(lab('조건부 밀도 $p(y \\mid x)$', 'conditional density $p(y \\mid x)$'))
ax.set_title(lab(f'$x = {X0}$ 에서의 조건부 분포: 손실마다 최적 답이 다르다',
                 f'conditional distribution at $x = {X0}$: each loss wants a different answer'), fontsize=10)
ax.legend(fontsize=8.5); show('conditional_density')

> **이 그림이 이 노트북의 출발점이다.** 세로선 세 개는 모두 **같은 분포**에서 나온 값이다.
> 어느 것이 옳은지는 분포가 정하지 않는다. **손실이 정한다.**

---
## 2. 세 손실로 실제로 적합해 보기

§1.2.1의 정리에 따라 각 $x$에서 **따로** 최소화하면 된다. 그래서 $x$를 구간으로 나누고 구간마다 상수 하나를 고른다.
그러면 세 손실의 경험적 최소해가 모두 **폐형식**이 되어 최적화기가 필요 없다.

| 손실 | 구간 안에서의 경험적 최소해 |
|---|---|
| 제곱오차 | 표본 평균 |
| 절대오차 | 표본 중앙값 |
| 0-1 (이산화된 $y$) | 최빈 구간 |

**최적화를 배제한 것이 요점이다.** 세 곡선이 갈라지는 것이 손실 때문인지 최적화 때문인지 헷갈릴 여지를 없앤다.
(경사하강으로 해도 같은 결과가 나오는 것은 6절에서 확인한다.)

In [ ]:
Y_EDGES = np.linspace(0.05, 25, 61)         # 0-1 손실을 위한 y 이산화
Y_CENT  = 0.5*(Y_EDGES[:-1] + Y_EDGES[1:])

def true_binned_mode(x, edges=Y_EDGES):
    m = mu(np.atleast_1d(x))[:, None]
    cdf = norm.cdf((np.log(edges)[None, :] - m)/S)
    return 0.5*(edges[:-1]+edges[1:])[np.argmax(np.diff(cdf, axis=1), axis=1)]

def pointwise_fit(x, y, nb=24, edges=Y_EDGES):
    bins = np.clip((x*nb).astype(int), 0, nb-1)
    centers = (np.arange(nb)+0.5)/nb
    out = np.full((3, nb), np.nan)
    for b in range(nb):
        s_ = y[bins == b]
        if len(s_) < 20:
            continue
        out[0, b] = s_.mean()
        out[1, b] = np.median(s_)
        cnt, _ = np.histogram(s_, bins=edges)
        out[2, b] = 0.5*(edges[np.argmax(cnt)] + edges[np.argmax(cnt)+1])
    return centers, out

n2 = 80_000 if FAST else 200_000
xs, ys = sample(n2, np.random.default_rng(1))
cen, fit = pointwise_fit(xs, ys)

print(f"n = {n2:,},  x 구간 24개")
print(f"  제곱오차 적합 vs 참 평균      : {np.nanmean(np.abs(fit[0]-true_mean(cen))):.4f}")
print(f"  절대오차 적합 vs 참 중앙값    : {np.nanmean(np.abs(fit[1]-true_median(cen))):.4f}")
print(f"  0-1 적합    vs 참 이산 최빈값 : {np.nanmean(np.abs(fit[2]-true_binned_mode(cen))):.4f}")
print( "  " + "-"*52)
print(f"  제곱오차 적합 vs 참 중앙값    : {np.nanmean(np.abs(fit[0]-true_median(cen))):.4f}  <- 안 맞는다")

In [ ]:
xg = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(xg, true_mean(xg),   '-',  color=CB[4], lw=1.2, alpha=0.9)
ax.plot(xg, true_median(xg), '-',  color=CB[3], lw=1.2, alpha=0.9)
ax.plot(xg, true_mode(xg),   '-',  color=CB[1], lw=1.2, alpha=0.9)
ax.plot(cen, fit[0], 'o', ms=5, color=CB[4], label=lab('제곱오차 적합', 'squared-loss fit'))
ax.plot(cen, fit[1], 's', ms=5, color=CB[3], label=lab('절대오차 적합', 'abs-loss fit'))
ax.plot(cen, fit[2], '^', ms=5, color=CB[1], label=lab('0-1 적합', '0-1 fit'))
ax.plot([], [], '-', color='0.4', label=lab('실선 = 폐형식 참값', 'lines = closed-form truth'))
ax.set_xlabel('$x$')
ax.set_ylabel(lab('예측값', 'prediction'))
ax.set_title(lab('같은 자료, 같은 모형, 다른 손실 — 세 곡선이 갈라진다',
                 'same data, same model, different losses'), fontsize=10)
ax.legend(fontsize=8); show('three_fits')

---
## 3. 핵심 실험 — 각 추정량은 **자기 목표에만** 수렴한다

여기가 이 노트북의 중심이다. $x = 0.25$ 하나에 집중해 네 가지 거리를 동시에 잰다.

| | $n \to \infty$ 에서 |
|---|---|
| 표본 평균 → 참 평균 | 0으로 가야 한다 |
| 표본 중앙값 → 참 중앙값 | 0으로 가야 한다 |
| **표본 평균 → 참 중앙값** | **0으로 가지 않아야 한다** |
| **표본 중앙값 → 참 평균** | **0으로 가지 않아야 한다** |

아래 두 줄이 $|{\text{평균}} - {\text{중앙값}}|$ 에서 평평해진다면, 그것은 자료가 부족해서가 아니라
**애초에 다른 것을 추정하고 있기 때문**이다. 로그–로그 축의 기울기로 판정한다.

In [ ]:
m_true, med_true = float(true_mean(X0)), float(true_median(X0))
gap = abs(m_true - med_true)

ns = np.unique(np.round(np.logspace(1.5, 4.5, 10)).astype(int))
T3 = 100 if FAST else 250
rng = np.random.default_rng(SEED)

keys = ['mean_to_mean', 'median_to_median', 'mean_to_median', 'median_to_mean']
E = {k: np.empty(len(ns)) for k in keys}
for i, n in enumerate(ns):
    z = np.exp(mu(X0) + S*rng.normal(size=(T3, n)))
    sm, smed = z.mean(axis=1), np.median(z, axis=1)
    E['mean_to_mean'][i]     = np.mean(np.abs(sm   - m_true))
    E['median_to_median'][i] = np.mean(np.abs(smed - med_true))
    E['mean_to_median'][i]   = np.mean(np.abs(sm   - med_true))
    E['median_to_mean'][i]   = np.mean(np.abs(smed - m_true))

print(f"x={X0}:  참 평균 {m_true:.4f}, 참 중앙값 {med_true:.4f}, 격차 {gap:.4f}\n")
for k in keys:
    s_, _ = np.polyfit(np.log(ns), np.log(E[k]), 1)
    print(f"  {k:18s}  기울기 {s_:+.3f}   n={ns[-1]:,}에서 {E[k][-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
styles = [('mean_to_mean',     CB[4], 'o-', lab('표본평균 → 참 평균',   'sample mean -> true mean')),
          ('median_to_median', CB[3], 's-', lab('표본중앙값 → 참 중앙값', 'sample median -> true median')),
          ('mean_to_median',   CB[4], 'o--', lab('표본평균 → 참 중앙값',  'sample mean -> true median')),
          ('median_to_mean',   CB[3], 's--', lab('표본중앙값 → 참 평균',  'sample median -> true mean'))]
for k, c, st, lb in styles:
    ax.loglog(ns, E[k], st, ms=4, color=c, label=lb)
ax.axhline(gap, color=CB[0], ls=':', lw=1.6,
           label=lab(f'참 격차 |평균−중앙값| = {gap:.2f}', f'true gap = {gap:.2f}'))
ax.set_xlabel(lab('표본 수 $n$ (개)', 'sample size $n$'))
ax.set_ylabel(lab('평균 절대 거리', 'mean absolute distance'))
ax.set_title(lab('실선은 내려가고 점선은 멈춘다 — 자료로 메울 수 없는 격차',
                 'solid lines fall, dashed lines plateau'), fontsize=10)
ax.legend(fontsize=7.5); show('convergence_to_own_target')

> ### 이 그림이 §1.2.2를 실험으로 확인한다
>
> 실선 둘은 기울기 $-1/2$로 내려간다. 점선 둘은 **기울기가 0**이고 참 격차에서 멈춘다.
>
> 점선의 높이는 **자료를 아무리 늘려도 줄지 않는다.** 이것은 추정 오차가 아니라 **다른 양을 추정한 결과**다.
> "손실은 최적화의 세부사항"이라는 흔한 오해가 여기서 반증된다 — 손실은 **무엇을 추정할지를 정하는 선택**이다.

---
## 4. 비대칭 강도를 훑기

$s \to 0$이면 로그정규가 대칭에 가까워지고 세 값이 겹친다. $s$를 키우면 갈라진다. 폐형식이 있으므로 실험이 필요 없다.

In [ ]:
ss = np.linspace(0.05, 1.5, 200)
fig, ax = plt.subplots(figsize=(5.6, 3.5))
ax.plot(ss, np.exp(ss**2/2), color=CB[4], label=lab('평균 / 중앙값', 'mean / median'))
ax.plot(ss, np.exp(-ss**2),  color=CB[1], label=lab('최빈값 / 중앙값', 'mode / median'))
ax.axhline(1, color=CB[0], lw=0.8)
ax.axvline(S, color=CB[5], ls='--', lw=1.4, label=lab(f'이 노트북 $s$ = {S}', f'this notebook $s$ = {S}'))
ax.set_xlabel(lab('비대칭 강도 $s$', 'asymmetry $s$'))
ax.set_ylabel(lab('중앙값 대비 배율', 'ratio to median'))
ax.set_title(lab('비대칭이 커질수록 세 최적해가 벌어진다',
                 'the three optima separate as asymmetry grows'), fontsize=10)
ax.legend(fontsize=8); show('asymmetry_sweep')

for s_ in [0.2, 0.5, 0.8, 1.2]:
    print(f"  s={s_:.1f}:  평균/중앙값 {np.exp(s_**2/2):.3f}배,  최빈값/중앙값 {np.exp(-s_**2):.3f}배")

---
## 5. 다봉 조건부 분포 — 제곱오차의 답이 **밀도가 없는 자리**에 놓인다

§1.2.2 ⚠︎(a)를 눈으로 확인한다. 조건부 분포를 두 봉우리의 혼합으로 바꾼다.

$$y \mid x \sim 0.35\,\mathcal{N}\big(-m(x),\ \sigma^2\big) + 0.65\,\mathcal{N}\big(+m(x),\ \sigma^2\big), \qquad m(x) = 2.5 + 0.8\sin(2\pi x)$$

평균은 폐형식 $(0.65-0.35)\,m(x)$이고, 중앙값은 수치적으로 푼다. 최빈값은 무거운 쪽 봉우리다.

In [ ]:
W1, SIG = 0.65, 0.4
def m_of(x):  return 2.5 + 0.8*np.sin(2*np.pi*np.asarray(x))
def bi_mean(x):   return (2*W1 - 1)*m_of(x)
def bi_mode(x):   return m_of(x)
def bi_density(t, x):
    mm = m_of(x)
    return (1-W1)*norm.pdf(t, -mm, SIG) + W1*norm.pdf(t, mm, SIG)
def bi_median(x):
    out = []
    for xi in np.atleast_1d(x):
        mm = m_of(xi)
        out.append(brentq(lambda t: (1-W1)*norm.cdf((t+mm)/SIG)
                                    + W1*norm.cdf((t-mm)/SIG) - 0.5, -12, 12))
    return np.array(out)

xg5 = np.linspace(0, 1, 200)
print("  x     평균    중앙값   최빈값   평균지점 밀도 / 최빈지점 밀도")
for xi in [0.0, 0.25, 0.5, 0.75]:
    me, md_, mo = bi_mean(xi), bi_median(xi)[0], bi_mode(xi)
    print(f"{xi:5.2f}  {me:6.3f}  {md_:6.3f}  {mo:6.3f}        {bi_density(me,xi)/bi_density(mo,xi):.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))

tt = np.linspace(-6, 6, 700)
axes[0].plot(tt, bi_density(tt, X0), color=CB[0], lw=1.6)
axes[0].fill_between(tt, bi_density(tt, X0), color=CB[2], alpha=0.18)
for v, c, lk, le in [(bi_mode(X0),      CB[1], '최빈값', 'mode'),
                     (bi_median(X0)[0], CB[3], '중앙값', 'median'),
                     (bi_mean(X0),      CB[4], '평균',   'mean')]:
    axes[0].axvline(float(v), color=c, lw=2, label=lab(f'{lk} {float(v):.2f}', f'{le} {float(v):.2f}'))
axes[0].set_xlabel('$y$')
axes[0].set_ylabel(lab('조건부 밀도', 'conditional density'))
axes[0].set_title(lab(f'$x={X0}$: 평균이 골짜기에 놓인다',
                      f'$x={X0}$: the mean lands in the valley'), fontsize=10)
axes[0].legend(fontsize=8)

axes[1].plot(xg5,  m_of(xg5), color=CB[1], lw=1.3, label=lab('최빈값 (봉우리)', 'mode'))
axes[1].plot(xg5, -m_of(xg5), color=CB[1], lw=1.3, ls=':', label=lab('가벼운 봉우리', 'minor peak'))
axes[1].plot(xg5, bi_median(xg5), color=CB[3], lw=1.6, label=lab('중앙값', 'median'))
axes[1].plot(xg5, bi_mean(xg5),   color=CB[4], lw=1.6, label=lab('평균 (제곱오차)', 'mean'))
axes[1].set_xlabel('$x$'); axes[1].set_ylabel(lab('예측값', 'prediction'))
axes[1].set_title(lab('제곱오차의 최적 곡선은 두 봉우리 사이를 지난다',
                      'the squared-loss optimum runs between the two modes'), fontsize=10)
axes[1].legend(fontsize=7.5)
show('bimodal')

> ### 왜 L2 손실이 흐릿한 출력을 만드는가
>
> $x = 0.25$에서 제곱오차의 최적 예측은 **밀도가 최빈값의 0.003% 수준인 자리**다.
> 모형이 실패한 것이 아니다. **손실이 요구한 답을 정확히 낸 것**이다.
>
> 영상 예측·초해상도에서 L2가 평균적으로 흐린 결과를 내는 이유가 이것이고,
> 5부(생성 모형)가 존재하는 이유의 절반이 이 그림에 있다.
> 25장에서 이것을 새로 설명할 필요가 없도록, 이 실험을 여기서 해 두는 것이다.

---
## 6. 경사하강으로 해도 같은가 — 그리고 분위수 손실 (★)

2절은 최적화를 배제하려고 구간별 폐형식을 썼다. 실제 학습에서도 같은지 확인한다.
RBF 기저 위의 선형 모형을 **같은 자료·같은 구조·같은 최적화기**로 두고 손실만 바꾼다.

함께 분위수 손실도 넣는다. §1.2.2 6절에서 절대오차의 일반화로 언급한 핀볼 손실이다.

$$\ell_\tau(a,y) = \max\big(\tau(y-a),\ (\tau-1)(y-a)\big) \quad \Longrightarrow \quad f^{*}(x) = \text{조건부 } \tau\text{-분위수}$$

로그정규의 참 분위수는 $\exp(\mu(x) + s\,\Phi^{-1}(\tau))$로 폐형식이므로 정답을 알고 검증할 수 있다.

In [ ]:
def rbf(x, cen, h):
    return np.exp(-0.5*((np.asarray(x)[:, None]-cen[None, :])/h)**2)

def fit_gd(x, y, loss, tau=0.5, iters=None, lr=0.5, nc=30):
    iters = iters or (1200 if FAST else 2500)
    cen = np.linspace(0, 1, nc); h = 1.0/nc
    P = np.hstack([rbf(x, cen, h), np.ones((len(x), 1))])
    ybar, ysd = y.mean(), y.std()
    yz = (y - ybar)/ysd
    w = np.zeros(P.shape[1]); v = np.zeros_like(w)
    for _ in range(iters):
        r = P @ w - yz
        if loss == 'mse':
            g = P.T @ (2*r)/len(x)
        elif loss == 'mae':
            g = P.T @ np.sign(r)/len(x)
        else:
            g = P.T @ np.where(r > 0, 1-tau, -tau)/len(x)
        v = 0.9*v + g
        w -= lr*v
    return lambda xq: (np.hstack([rbf(xq, cen, h), np.ones((len(np.asarray(xq)), 1))]) @ w)*ysd + ybar

n6 = 20_000 if FAST else 50_000
x6, y6 = sample(n6, np.random.default_rng(3))
xq = np.linspace(0.02, 0.98, 60)

g_mse, g_mae = fit_gd(x6, y6, 'mse'), fit_gd(x6, y6, 'mae')
print("경사하강 적합 (구조·자료·최적화기 동일, 손실만 다름)")
print(f"  MSE -> 참 평균   : {np.mean(np.abs(g_mse(xq)-true_mean(xq))):.4f}")
print(f"  MSE -> 참 중앙값 : {np.mean(np.abs(g_mse(xq)-true_median(xq))):.4f}   <- 안 맞는다")
print(f"  MAE -> 참 중앙값 : {np.mean(np.abs(g_mae(xq)-true_median(xq))):.4f}")
print(f"  MAE -> 참 평균   : {np.mean(np.abs(g_mae(xq)-true_mean(xq))):.4f}   <- 안 맞는다")

TAUS = [0.1, 0.5, 0.9]
g_q = {t: fit_gd(x6, y6, 'pinball', tau=t) for t in TAUS}
print("\n분위수 손실")
for t in TAUS:
    q_true = np.exp(mu(xq) + S*norm.ppf(t))
    rel = np.mean(np.abs(g_q[t](xq)-q_true)/q_true)
    print(f"  tau={t}: 참 분위수 대비 평균 상대오차 {rel:.3%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6), sharex=True)

axes[0].plot(xg, true_mean(xg),   '-', color=CB[4], lw=1.1)
axes[0].plot(xg, true_median(xg), '-', color=CB[3], lw=1.1)
axes[0].plot(xq, g_mse(xq), 'o', ms=3.5, color=CB[4], label=lab('MSE 경사하강', 'MSE by GD'))
axes[0].plot(xq, g_mae(xq), 's', ms=3.5, color=CB[3], label=lab('MAE 경사하강', 'MAE by GD'))
axes[0].plot([], [], '-', color='0.4', label=lab('실선 = 폐형식 참값', 'lines = truth'))
axes[0].set_title(lab('경사하강으로 해도 결과는 같다', 'gradient descent gives the same answer'), fontsize=10)
axes[0].set_ylabel(lab('예측값', 'prediction')); axes[0].set_xlabel('$x$')
axes[0].legend(fontsize=7.5)

for t, c in zip(TAUS, [CB[2], CB[3], CB[5]]):
    axes[1].plot(xg, np.exp(mu(xg) + S*norm.ppf(t)), '-', color=c, lw=1.0)
    axes[1].plot(xq, g_q[t](xq), 'o', ms=3, color=c,
                 label=lab(f'$\\tau$ = {t}', f'$\\tau$ = {t}'))
axes[1].fill_between(xg, np.exp(mu(xg)+S*norm.ppf(0.1)), np.exp(mu(xg)+S*norm.ppf(0.9)),
                     color=CB[2], alpha=0.12)
axes[1].set_title(lab('분위수 손실은 예측 구간을 준다', 'pinball loss gives prediction bands'), fontsize=10)
axes[1].set_xlabel('$x$'); axes[1].legend(fontsize=7.5)
show('gd_and_quantiles')

---
## 7. ⚠︎ 최빈값만 다른 대접을 받는다

평균과 중앙값은 척도에 대해 잘 정의되지만 **최빈값은 이산화에 의존한다.**
연속 $y$에 0-1 손실은 그대로 적용되지 않고, §1.2.2 3절이 유한 $\mathcal{Y}$를 전제한 것이 이 때문이다.
구간 수를 바꾸면 답이 달라지는 것을 직접 확인한다.

In [ ]:
print(f"x={X0}에서 연속 최빈값(폐형식) = {float(true_mode(X0)):.4f}\n")
print("  구간 수    구간 폭    이산 최빈값")
for K in [11, 21, 41, 81, 161, 321]:
    ed = np.linspace(0.05, 25, K+1)
    print(f"  {K:>5}    {(ed[1]-ed[0]):7.4f}    {true_binned_mode(X0, ed)[0]:.4f}")
print("\n-> 전체 추세로는 연속 최빈값에 접근하지만 단조적이지 않다.")
print("   구간 '폭'뿐 아니라 '위치'도 답을 바꾸기 때문이다 (K=21과 K=41을 비교할 것).")
print("   게다가 실제 자료에서는 구간이 잘아질수록 구간당 표본이 줄어 분산이 커진다.")
print("   최빈값은 평균/중앙값과 달리 '무엇을 최빈값이라 부를지'를 먼저 정해야 하는 양이다.")

---
## 8. 자기 점검

1. 3절에서 $S$를 $0.1$로 줄이면 점선 두 개는 어떻게 되는가? **사라지는가, 낮아지는가?** 예측한 뒤 손잡이를 바꿔 확인하라.
2. 5절에서 두 봉우리의 가중치를 $0.5 / 0.5$로 바꾸면 중앙값은 어디로 가는가? 평균은?
3. 6절에서 $\tau = 0.5$인 핀볼 손실과 절대오차가 같은 답을 주는 것을 확인하라. 왜 같은가?
4. 2절에서 $x$ 구간 수를 24에서 200으로 늘리면 세 곡선은 참값에 더 가까워지는가? 어느 곡선이 가장 먼저 나빠지는가?

In [ ]:
# 자기 점검 3의 확인
g_half = fit_gd(x6, y6, 'pinball', tau=0.5)
d = np.mean(np.abs(g_half(xq) - g_mae(xq)))
print(f"핀볼(tau=0.5)와 절대오차 적합의 평균 차이: {d:.6f}")
print("-> l_0.5(a,y) = |y-a|/2 이므로 목적함수가 상수배 차이. 최소해는 같다 (§1.1.1의 척도 무의미성)")

---
## 9. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `S` | 0절 | 0.8 | 비대칭 강도. 0.1이면 세 곡선이 거의 겹치고, 1.5면 크게 갈라진다 |
| `X0` | 1절 | 0.25 | 자세히 볼 입력 지점 |
| `Y_EDGES` | 2절 | 61개 | 0-1 손실의 이산화. 7절의 실험 대상 |
| `W1`, `SIG` | 5절 | 0.65, 0.4 | 봉우리 가중치와 폭. `W1=0.5`면 평균과 중앙값이 모두 0 |
| `TAUS` | 6절 | [0.1,0.5,0.9] | 분위수. [0.01, 0.99]로 넓히면 꼬리 추정의 어려움이 보인다 |
| `SAVE_PDF` | 0절 | False | True면 그림을 벡터 PDF로 저장 |
| `FAST` | 0절 | False | True면 15초에 완주 |

**권하는 첫 실험** — `S = 0.05`로 두고 전체 재실행. 3절의 점선 두 개가 **사라지지 않고 아주 낮은 높이에서 여전히 평평합니다.**
격차가 작아졌을 뿐 여전히 다른 양을 추정하고 있다는 것 — 이것이 "대칭 분포에서는 손실이 중요하지 않다"는 오해를 깨는 가장 빠른 방법입니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t_start:.1f}초")